# 06 - Run Classical Baseline Optimizers (CMA-ES, DE, PSO)

This notebook:
1. Defines classical baseline algorithms: **CMA-ES**, **Differential Evolution (DE)**, and **Particle Swarm Optimization (PSO)**.
2. Runs each baseline **N=10 independent times** on target BBOB problems across multiple dimensions and noise levels.
3. Uses configured noise strategies (`MultiplicativeNoiseStrategy` / `NoNoiseStrategy`).
4. Attaches IOH Analyzer via `problem.attach_analyzer(...)` to output IOH `.dat` performance files to `data/ioh_logs/{dim}D/std_{noise_std}/f{p_id}/{cmaes|de|pso}/`.

In [3]:
import sys
import numpy as np
import cma
from scipy.optimize import differential_evolution
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from infra.problems.bbob import BBOBProblem
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy

# ── Experiment Configuration ─────────────────────────────
TARGET_PROBLEMS = [1, 8, 11, 15, 21]
DIMS            = [2, 3]             # Dimensions to evaluate
NOISE_STDS      = [0.0, 0.05, 0.1]  # Noise std levels to evaluate
IOH_LOGS_DIR    = PROJECT_ROOT / 'data' / 'ioh_logs'
N_RUNS          = 10                 # Independent runs per config
BUDGET          = 100000             # Function evaluation budget
# ─────────────────────────────────────────────────────────

print(f'Target Problems: {TARGET_PROBLEMS}')
print(f'Target Dimensions: {DIMS}')
print(f'Target Noise STDs: {NOISE_STDS}')
print(f'Runs per problem: {N_RUNS}')
print(f'Budget: {BUDGET} evaluations')

Target Problems: [1, 8, 11, 15, 21]
Target Dimensions: [2, 3]
Target Noise STDs: [0.0, 0.05, 0.1]
Runs per problem: 10
Budget: 100000 evaluations


## 1. Algorithm Implementations

In [4]:
def run_cmaes(problem, dim: int, budget: int):
    """Run CMA-ES algorithm on problem with specified evaluation budget."""
    x0 = [0.0] * dim
    sigma0 = 2.0
    opts = {'bounds': [-5.0, 5.0], 'verbose': -9, 'maxfevals': budget}
    es = cma.CMAEvolutionStrategy(x0, sigma0, opts)
    while not es.stop():
        solutions = es.ask()
        es.tell(solutions, [problem(x) for x in solutions])
        if problem.evaluations >= budget:
            break
    return es.result.xbest, es.result.fbest

def run_de(problem, dim: int, budget: int):
    """Run Differential Evolution (DE) on problem with specified evaluation budget."""
    bounds = [(-5.0, 5.0)] * dim
    maxiter = max(1, budget // (15 * dim))
    
    def obj_fn(x):
        if problem.evaluations >= budget:
            raise StopIteration('Budget exhausted')
        return problem(x)
        
    try:
        res = differential_evolution(obj_fn, bounds, maxiter=maxiter, seed=None)
        return res.x, res.fun
    except StopIteration:
        return problem.optimum_x, problem.true_optimum

def run_pso(problem, dim: int, budget: int, n_particles: int = 30, w: float = 0.729, c1: float = 1.49445, c2: float = 1.49445):
    """Run Particle Swarm Optimization (PSO) on problem with specified evaluation budget."""
    lb, ub = problem.lb, problem.ub
    X = np.random.uniform(lb, ub, (n_particles, dim))
    V = np.random.uniform(-abs(ub - lb), abs(ub - lb), (n_particles, dim)) * 0.1
    
    pbest_X = X.copy()
    pbest_y = np.array([problem(x) for x in X])
    
    gbest_idx = np.argmin(pbest_y)
    gbest_X = pbest_X[gbest_idx].copy()
    gbest_y = pbest_y[gbest_idx]
    
    evals = n_particles
    while evals < budget:
        r1 = np.random.rand(n_particles, dim)
        r2 = np.random.rand(n_particles, dim)
        V = w * V + c1 * r1 * (pbest_X - X) + c2 * r2 * (gbest_X - X)
        X = np.clip(X + V, lb, ub)
        
        for i in range(n_particles):
            if evals >= budget:
                break
            y = problem(X[i])
            evals += 1
            if y < pbest_y[i]:
                pbest_y[i] = y
                pbest_X[i] = X[i].copy()
                if y < gbest_y:
                    gbest_y = y
                    gbest_X = X[i].copy()
                    
    return gbest_X, gbest_y

## 2. Benchmark Execution Loop

In [5]:
BASELINES = {
    'cmaes': run_cmaes,
    'de': run_de,
    'pso': run_pso,
}

for dim in DIMS:
    for noise_std in NOISE_STDS:
        for p_id in TARGET_PROBLEMS:
            # Standardized directory structure: data/ioh_logs/{dim}D/std_{noise_std}/f{p_id}/
            out_dir = IOH_LOGS_DIR / f'{dim}D' / f'std_{noise_std}' / f'f{p_id}'
            out_dir.mkdir(parents=True, exist_ok=True)
            
            for algo_name, runner_fn in BASELINES.items():
                folder_name = algo_name
                print()
                print(f'=== Running {algo_name.upper()} on f{p_id} ({dim}D, noise={noise_std}, N={N_RUNS} runs, Budget={BUDGET}) ===')
                
                noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
                problem = BBOBProblem(
                    problem_id=p_id,
                    dim=dim,
                    instance_id=1,
                    noise_strategy=noise_strat,
                )
                
                # Attach IOH logger once for all N runs
                problem.attach_analyzer(
                    log_dir=out_dir,
                    folder_name=folder_name,
                    algorithm_name=algo_name.upper(),
                    algorithm_info=f'dim={dim}, noise_std={noise_std}',
                )
                
                for run_idx in range(1, N_RUNS + 1):
                    problem.reset()
                    try:
                        best_x, best_y = runner_fn(problem, dim, BUDGET)
                    except Exception as e:
                        print(f'  Run {run_idx:2d}/{N_RUNS} error: {e}')
                        continue
                        
                    clean_val = problem.eval_clean(problem.clip(best_x))
                    clean_err = abs(clean_val - problem.true_optimum)
                    print(f'  Run {run_idx:2d}/{N_RUNS}: final clean error = {clean_err:.6e}')
                    
                # Safely close logger after all N runs complete
                problem.close_logger()
                print(f'  Saved IOH logs for {algo_name.upper()} to {out_dir / folder_name}')

print('\nAll baseline runs complete!')


=== Running CMAES on f1 (2D, noise=0.0, N=10 runs, Budget=100000) ===
  Run  1/10: final clean error = 0.000000e+00
  Run  2/10: final clean error = 0.000000e+00
  Run  3/10: final clean error = 0.000000e+00
  Run  4/10: final clean error = 0.000000e+00
  Run  5/10: final clean error = 0.000000e+00
  Run  6/10: final clean error = 0.000000e+00
  Run  7/10: final clean error = 0.000000e+00
  Run  8/10: final clean error = 0.000000e+00
  Run  9/10: final clean error = 0.000000e+00
  Run 10/10: final clean error = 0.000000e+00
  Saved IOH logs for CMAES to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/ioh_logs/2D/std_0.0/f1/cmaes

=== Running DE on f1 (2D, noise=0.0, N=10 runs, Budget=100000) ===
  Run  1/10: final clean error = 4.263256e-14
  Run  2/10: final clean error = 2.842171e-14
  Run  3/10: final clean error = 5.684342e-14
  Run  4/10: final clean error = 2.842171e-14
  Run  5/10: final clean error = 2.842171e-14
  Run  6/10: final clean error = 1.278977e-13
  Run  7/10: final 

KeyboardInterrupt: 